# Genetic ALO — Toàn bộ dự án trên Google Colab

Notebook duy nhất cho toàn bộ quy trình: clone source từ GitHub, cài thư viện,
xem dataset Excel, chạy kiểm thử, tạo thời khóa biểu bằng GA + Repair + SLS,
phân tích độ nhạy, benchmark và mở giao diện Streamlit.

## Cách chạy nhanh khi trình bày

1. Chọn **Runtime → Run all**.
2. Cấp quyền Google Drive để lưu output.
3. Chờ phần production tạo lịch và phần cuối cấp link **MỞ WEB DEMO**.

Pytest, sensitivity và benchmark mặc định tắt để buổi demo không phải chờ lâu.
Có thể bật từng công tắc trong các phần tương ứng khi cần báo cáo chi tiết.

In [ ]:
#@title Clone dự án từ GitHub và cài môi trường Colab
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
from pathlib import Path
import shutil
import subprocess
import sys

THU_MUC_COLAB_DRIVE = Path("/content/drive/MyDrive/Genetic_ALO_Colab")
THU_MUC_COLAB_DRIVE.mkdir(parents=True, exist_ok=True)
THU_MUC_DU_AN = Path("/content/genetic-alo")
THU_MUC_KET_QUA_DRIVE = THU_MUC_COLAB_DRIVE / "latest_outputs"
REPOSITORY_URL = "https://github.com/duktrung05/genetic-alo.git"

shutil.rmtree(THU_MUC_DU_AN, ignore_errors=True)
subprocess.run(
    [
        "git", "clone", "--depth", "1", "--branch", "main",
        REPOSITORY_URL, str(THU_MUC_DU_AN),
    ],
    check=True,
)

# Khôi phục output mới nhất từ Drive nếu notebook trước đã tạo kết quả.
if THU_MUC_KET_QUA_DRIVE.is_dir():
    shutil.copytree(
        THU_MUC_KET_QUA_DRIVE,
        THU_MUC_DU_AN / "outputs",
        dirs_exist_ok=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "--disable-pip-version-check", "-r",
        str(THU_MUC_DU_AN / "requirements.txt"),
    ],
    check=True,
)

os.chdir(THU_MUC_DU_AN)
if str(THU_MUC_DU_AN) not in sys.path:
    sys.path.insert(0, str(THU_MUC_DU_AN))

def dong_bo_ket_qua() -> Path:
    """Copy all current outputs to Drive so another notebook can reuse them."""
    THU_MUC_KET_QUA_DRIVE.mkdir(parents=True, exist_ok=True)
    shutil.copytree(
        THU_MUC_DU_AN / "outputs",
        THU_MUC_KET_QUA_DRIVE,
        dirs_exist_ok=True,
    )
    return THU_MUC_KET_QUA_DRIVE

print(f"✅ Đã clone dự án tại: {THU_MUC_DU_AN}")
print(f"✅ Python: {sys.version.split()[0]}")
subprocess.run(["git", "log", "-1", "--oneline"], cwd=THU_MUC_DU_AN, check=True)
print("✅ Dataset Excel nằm trong data/instances.")

# 01 — Kiểm thử source code và dữ liệu

Notebook này khôi phục dự án từ Google Drive, xác minh hai dataset Excel, import
các module chính và chạy toàn bộ pytest. Chạy **Runtime → Run all**.

In [ ]:
#@title Kiểm tra cấu trúc và tính hợp lệ của dataset
from dataset import DatasetValidator, ExcelDatasetLoader

cac_tep_bat_buoc = [
    THU_MUC_DU_AN / "main.py",
    THU_MUC_DU_AN / "main_benchmark.py",
    THU_MUC_DU_AN / "ui_app.py",
    THU_MUC_DU_AN / "data/instances/instance_easy.xlsx",
    THU_MUC_DU_AN / "data/instances/instance_medium.xlsx",
]
tep_thieu = [str(path) for path in cac_tep_bat_buoc if not path.is_file()]
if tep_thieu:
    raise FileNotFoundError("Thiếu file bắt buộc:\n- " + "\n- ".join(tep_thieu))

for ten in ("easy", "medium"):
    path = THU_MUC_DU_AN / f"data/instances/instance_{ten}.xlsx"
    dataset = ExcelDatasetLoader.load_and_validate(str(path))
    report = DatasetValidator.validate_report(dataset)
    if not report["valid"]:
        raise ValueError(f"Dataset {ten} không hợp lệ: {report['errors']}")
    print(f"✅ Dataset {ten.upper()} hợp lệ: {path.name}")

In [ ]:
#@title Xem trực tiếp dữ liệu Excel
import pandas as pd
from IPython.display import display

XEM_DATASET = "easy" #@param ["easy", "medium"]
SO_DONG_MOI_SHEET = 5 #@param {type:"integer"}

duong_dan_excel = THU_MUC_DU_AN / f"data/instances/instance_{XEM_DATASET}.xlsx"
tep_excel = pd.ExcelFile(duong_dan_excel)
print(f"Dataset thật: {duong_dan_excel}")
print("Các sheet:", tep_excel.sheet_names)

for ten_sheet in tep_excel.sheet_names:
    print(f"\n--- {ten_sheet} ---")
    display(pd.read_excel(duong_dan_excel, sheet_name=ten_sheet).head(SO_DONG_MOI_SHEET))

In [ ]:
#@title Chạy toàn bộ pytest
CHAY_TOAN_BO_TEST = False #@param {type:"boolean"}

if CHAY_TOAN_BO_TEST:
    subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=THU_MUC_DU_AN,
        check=True,
    )
    print("✅ Toàn bộ kiểm thử đã pass.")
else:
    print("ℹ️ Đã bỏ qua pytest.")

# 02 — Chạy Genetic Algorithm và xuất thời khóa biểu

Notebook chạy luồng production **GA + Repair + Soft Local Search**, xuất Excel,
JSON và biểu đồ hội tụ. Kết quả được đồng bộ sang Google Drive để notebook demo
sử dụng đúng lịch vừa tạo.

In [ ]:
#@title Cấu hình và chạy thuật toán
TEN_DATASET = "easy" #@param ["easy", "medium"]
SEED = 42 #@param {type:"integer"}
KICH_THUOC_QUAN_THE = 60 #@param {type:"integer"}
NGAN_SACH_DANH_GIA = 1000 #@param {type:"integer"}
DUNG_SOFT_LOCAL_SEARCH = True #@param {type:"boolean"}
HO_SO_TRONG_SO = "balanced" #@param ["student_centric", "balanced", "resource_centric"]

duong_dan_input = THU_MUC_DU_AN / f"data/instances/instance_{TEN_DATASET}.xlsx"
thu_muc_ket_qua = THU_MUC_DU_AN / "outputs/production"
thu_muc_ket_qua.mkdir(parents=True, exist_ok=True)
duong_dan_output = thu_muc_ket_qua / "best_timetable.xlsx"

lenh_chay = [
    sys.executable,
    "main.py",
    "--input", str(duong_dan_input),
    "--output", str(duong_dan_output),
    "--seed", str(SEED),
    "--population-size", str(KICH_THUOC_QUAN_THE),
    "--search-evaluation-budget", str(NGAN_SACH_DANH_GIA),
    "--weight-profile", HO_SO_TRONG_SO,
]
if DUNG_SOFT_LOCAL_SEARCH:
    lenh_chay.append("--soft-local-search")

print("Lệnh thực thi:", " ".join(lenh_chay))
subprocess.run(lenh_chay, cwd=THU_MUC_DU_AN, check=True)
vi_tri_drive = dong_bo_ket_qua()
print(f"✅ File thời khóa biểu: {duong_dan_output}")
print(f"✅ Đã đồng bộ output sang: {vi_tri_drive}")

In [ ]:
#@title Xem trước Excel và biểu đồ hội tụ
import pandas as pd
from IPython.display import Image, display
from openpyxl import load_workbook

if not duong_dan_output.is_file():
    raise FileNotFoundError("Chưa có file kết quả. Hãy chạy cell thuật toán trước.")

workbook = load_workbook(duong_dan_output, read_only=True, data_only=True)
print("Các sheet:", workbook.sheetnames)
ten_sheet = workbook.sheetnames[0]
workbook.close()

display(pd.read_excel(duong_dan_output, sheet_name=ten_sheet).head(20))

ma_phuong_phap = "ga_repair_sls" if DUNG_SOFT_LOCAL_SEARCH else "ga_repair"
duong_dan_bieu_do = thu_muc_ket_qua / f"convergence_{ma_phuong_phap}.png"
if duong_dan_bieu_do.is_file():
    display(Image(filename=str(duong_dan_bieu_do)))

In [ ]:
#@title Tải toàn bộ kết quả về máy (tùy chọn)
TAI_KET_QUA = False #@param {type:"boolean"}

if TAI_KET_QUA:
    from google.colab import files
    tep_zip = shutil.make_archive(
        "/content/ket_qua_genetic_alo",
        "zip",
        root_dir=THU_MUC_DU_AN / "outputs",
    )
    files.download(tep_zip)
else:
    print("ℹ️ Kết quả đã nằm trên Google Drive; bật TAI_KET_QUA nếu muốn tải ZIP.")

# 03 — Phân tích độ nhạy trọng số

Notebook chạy thí nghiệm Phase 1.1 trên ba cấu hình trọng số và ba seed. Kết quả
gồm `raw_runs.csv` và `summary.json`, sau đó được lưu sang Google Drive.

In [ ]:
#@title Chạy phân tích độ nhạy
CHAY_PHAN_TICH_DO_NHAY = False #@param {type:"boolean"}
TEN_DATASET = "easy" #@param ["easy", "medium"]

thu_muc_do_nhay = THU_MUC_DU_AN / "outputs/benchmark/phase1_1_weight_sensitivity"
if CHAY_PHAN_TICH_DO_NHAY:
    subprocess.run(
        [
            sys.executable,
            "scripts/run_phase1_1_sensitivity.py",
            "--input", str(THU_MUC_DU_AN / f"data/instances/instance_{TEN_DATASET}.xlsx"),
            "--output-dir", str(thu_muc_do_nhay),
        ],
        cwd=THU_MUC_DU_AN,
        check=True,
    )
    print(f"✅ Đã đồng bộ sang: {dong_bo_ket_qua()}")
else:
    print("ℹ️ Đã bỏ qua phân tích độ nhạy.")

In [ ]:
#@title Hiển thị bảng tổng hợp
import json
import pandas as pd
from IPython.display import display

summary_path = thu_muc_do_nhay / "summary.json"
if not summary_path.is_file():
    raise FileNotFoundError("Chưa có summary.json. Hãy chạy cell phân tích trước.")

payload = json.loads(summary_path.read_text(encoding="utf-8"))
rows = []
for profile, values in payload["summary"].items():
    rows.append({
        "profile": profile,
        "runs": values["runs"],
        "feasible_runs": values["feasible_runs"],
        "mean_total_soft_score": values["mean_total_soft_score"],
        "mean_runtime_seconds": values["mean_runtime_seconds"],
    })
display(pd.DataFrame(rows))

# 04 — Benchmark so sánh phương pháp

Phần đầu chạy benchmark nhanh phục vụ trình bày. Phần cuối là benchmark chính
thức gồm 60 lượt chạy; mặc định tắt để tránh vô tình tốn nhiều thời gian.
Checkpoint được đồng bộ sang Drive nên có thể tiếp tục sau khi Colab ngắt phiên.

In [ ]:
#@title Benchmark nhanh
CHAY_BENCHMARK_NHANH = False #@param {type:"boolean"}
CAC_PHUONG_PHAP = "ga_repair_sls,ga_repair,ga" #@param {type:"string"}
CAC_SEED = "0-2" #@param {type:"string"}
TEN_DATASET = "easy" #@param ["easy", "medium"]

if CHAY_BENCHMARK_NHANH:
    subprocess.run(
        [
            sys.executable, "main_benchmark.py",
            "--mode", "fast",
            "--methods", CAC_PHUONG_PHAP,
            "--seeds", CAC_SEED,
            "--data-source", "excel",
            "--input", str(THU_MUC_DU_AN / f"data/instances/instance_{TEN_DATASET}.xlsx"),
            "--experiment-name", "colab_fast",
        ],
        cwd=THU_MUC_DU_AN,
        check=True,
    )
    print(f"✅ Đã đồng bộ sang: {dong_bo_ket_qua()}")
else:
    print("ℹ️ Đã bỏ qua benchmark nhanh.")

In [ ]:
#@title Xem kết quả benchmark nhanh
import pandas as pd
from IPython.display import display

thu_muc_benchmark_nhanh = THU_MUC_DU_AN / "outputs/benchmark/colab_fast"
tep_summary = thu_muc_benchmark_nhanh / "summary.csv"
if tep_summary.is_file():
    display(pd.read_csv(tep_summary))
else:
    print("Kết quả được tạo trong:", thu_muc_benchmark_nhanh)
    print([str(path.relative_to(THU_MUC_DU_AN)) for path in thu_muc_benchmark_nhanh.glob("*")])

In [ ]:
#@title Benchmark chính thức 60 lượt chạy (tùy chọn)
CHAY_BENCHMARK_CUOI = False #@param {type:"boolean"}
CHAY_LAI_TU_DAU = False #@param {type:"boolean"}

if CHAY_BENCHMARK_CUOI:
    lenh = [sys.executable, "scripts/run_final_benchmark.py"]
    if CHAY_LAI_TU_DAU:
        lenh.append("--fresh")
    subprocess.run(lenh, cwd=THU_MUC_DU_AN, check=True)
    print(f"✅ Benchmark hoàn tất; đã đồng bộ sang: {dong_bo_ket_qua()}")
else:
    print("ℹ️ Benchmark 60 lượt đang tắt. Chỉ bật khi có đủ thời gian chạy.")

# 05 — Mở giao diện Streamlit giống web demo

Notebook khôi phục đúng `ui_app.py` và output mới nhất, khởi động Streamlit trong
máy ảo Colab rồi tạo một đường dẫn TryCloudflare tạm thời. Không cần tài khoản
Cloudflare hoặc token. Hãy giữ phiên Colab hoạt động trong lúc trình bày.

> Đường dẫn là công khai và tạm thời; chỉ chia sẻ trong buổi demo.

In [ ]:
#@title Kiểm tra dữ liệu dùng cho demo
cac_tep_demo = [
    THU_MUC_DU_AN / "ui_app.py",
    THU_MUC_DU_AN / "data/instances/instance_easy.xlsx",
    THU_MUC_DU_AN / "outputs/production/schedule_query_data.json",
    THU_MUC_DU_AN / "outputs/production/best_timetable_metadata.json",
]
tep_thieu = [str(path) for path in cac_tep_demo if not path.is_file()]
if tep_thieu:
    raise FileNotFoundError(
        "Thiếu dữ liệu demo. Hãy chạy notebook 02 trước:\n- " + "\n- ".join(tep_thieu)
    )
print("✅ Dữ liệu demo đã sẵn sàng.")

In [ ]:
#@title Khởi động Streamlit và tạo đường dẫn demo
MO_GIAO_DIEN_STREAMLIT = True #@param {type:"boolean"}

import re
import platform
import time
import urllib.request
from IPython.display import HTML, display

if not MO_GIAO_DIEN_STREAMLIT:
    print("ℹ️ Đã bỏ qua việc mở giao diện.")
else:
    # Dừng đúng các tiến trình do notebook này tạo nếu chạy lại cell.
    for ten_bien in ("tien_trinh_streamlit", "tien_trinh_tunnel"):
        tien_trinh_cu = globals().get(ten_bien)
        if tien_trinh_cu is not None and tien_trinh_cu.poll() is None:
            tien_trinh_cu.terminate()

    tep_log_streamlit = Path("/content/streamlit_colab.log")
    tep_log_tunnel = Path("/content/cloudflare_tunnel.log")
    log_streamlit = open(tep_log_streamlit, "w", encoding="utf-8")

    tien_trinh_streamlit = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", "ui_app.py",
            "--server.headless=true",
            "--server.address=0.0.0.0",
            "--server.port=8501",
            "--browser.gatherUsageStats=false",
        ],
        cwd=THU_MUC_DU_AN,
        env={**os.environ, "GA_DEMO_EVALUATION_BUDGET": "100"},
        stdout=log_streamlit,
        stderr=subprocess.STDOUT,
        text=True,
    )

    # Chờ endpoint health thay vì ngủ một khoảng cố định.
    for _ in range(60):
        if tien_trinh_streamlit.poll() is not None:
            break
        try:
            with urllib.request.urlopen(
                "http://127.0.0.1:8501/_stcore/health", timeout=2
            ) as response:
                if response.status == 200 and response.read().decode().strip() == "ok":
                    break
        except Exception:
            time.sleep(1)
    else:
        raise TimeoutError("Streamlit không phản hồi health check sau 60 giây.")

    if tien_trinh_streamlit.poll() is not None:
        log_streamlit.flush()
        raise RuntimeError(tep_log_streamlit.read_text(errors="replace"))

    # Dùng cloudflared binary trực tiếp để không phụ thuộc Node.js/Wrangler.
    kien_truc = platform.machine().lower()
    ten_kien_truc = {
        "x86_64": "amd64",
        "amd64": "amd64",
        "aarch64": "arm64",
        "arm64": "arm64",
    }.get(kien_truc)
    if ten_kien_truc is None:
        raise RuntimeError(f"Kiến trúc Colab chưa được hỗ trợ: {kien_truc}")

    tep_cloudflared = Path("/content/cloudflared")
    url_cloudflared = (
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        f"cloudflared-linux-{ten_kien_truc}"
    )
    if not tep_cloudflared.is_file():
        print("Đang tải cloudflared chính thức...")
        urllib.request.urlretrieve(url_cloudflared, tep_cloudflared)
        tep_cloudflared.chmod(0o755)
    subprocess.run([str(tep_cloudflared), "--version"], check=True)

    url_cong_khai = None
    for lan_thu in range(1, 3):
        log_tunnel = open(tep_log_tunnel, "w", encoding="utf-8")
        tien_trinh_tunnel = subprocess.Popen(
            [
                str(tep_cloudflared), "tunnel", "--no-autoupdate",
                "--url", "http://127.0.0.1:8501",
            ],
            stdout=log_tunnel,
            stderr=subprocess.STDOUT,
            text=True,
        )
        for _ in range(90):
            time.sleep(1)
            log_tunnel.flush()
            noi_dung = tep_log_tunnel.read_text(encoding="utf-8", errors="replace")
            ket_qua = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", noi_dung)
            if ket_qua:
                url_cong_khai = ket_qua.group(0)
                break
            if tien_trinh_tunnel.poll() is not None:
                break
        if url_cong_khai:
            break
        if tien_trinh_tunnel.poll() is None:
            tien_trinh_tunnel.terminate()
        print(f"Thử tạo tunnel lần {lan_thu} chưa thành công; đang thử lại...")

    if not url_cong_khai:
        print(tep_log_tunnel.read_text(encoding="utf-8", errors="replace"))
        raise RuntimeError("Không tạo được link demo sau 2 lần thử.")

    print("✅ Streamlit health check: OK")
    print("✅ Link demo:", url_cong_khai)
    display(HTML(f'<a href="{url_cong_khai}" target="_blank" '
                 'style="font-size:20px;font-weight:bold">MỞ WEB DEMO</a>'))

## Phương án dự phòng khi mạng tunnel không ổn định

Notebook 02 vẫn chạy thuật toán, hiển thị bảng thời khóa biểu và biểu đồ ngay
trong Colab. Khi thi, hãy mở sẵn cả notebook 02 và 05; nếu link tạm thời gặp lỗi,
trình bày kết quả trực tiếp từ notebook 02.